---
## ⚙️ Install Dependencies

In [ ]:
!pip install -q kaggle shap xgboost transformers datasets accelerate scikit-learn pandas numpy matplotlib seaborn wordcloud

In [ ]:
from google.colab import files
import os

# Upload kaggle.json
print("Upload your kaggle.json file:")
uploaded = files.upload()

# Place in correct directory
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Download dataset
!kaggle datasets download -d shivamb/real-or-fake-fake-jobposting-prediction --unzip
print("✅ Dataset downloaded!")

### Step 1B: Load and Explore Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('fake_job_postings.csv')

print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nClass distribution:")
print(df['fraudulent'].value_counts())
print(f"\nFraud rate: {df['fraudulent'].mean()*100:.1f}%")

df.head(3)

In [ ]:
# Visualize class imbalance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
counts = df['fraudulent'].value_counts()
axes[0].bar(['Real (0)', 'Fake (1)'], counts.values, color=['#2ecc71', '#e74c3c'], width=0.5)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

# Missing values heatmap
missing = df.isnull().mean().sort_values(ascending=False) * 100
axes[1].barh(missing.index, missing.values, color='#3498db')
axes[1].set_title('Missing Values (%)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('% Missing')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: class_distribution.png")

### Step 1C: Preprocessing — Clean Text

In [ ]:
import re
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words('english'))

def clean_text(text):
    """Clean a single text string."""
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)          # Remove HTML tags
    text = re.sub(r'http\S+|www\.\S+', ' ', text) # Remove URLs
    text = re.sub(r'[^a-zA-Z\s!?]', ' ', text)   # Keep letters + !?
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    # Remove stopwords but keep short words that might be meaningful
    tokens = [w for w in text.split() if w not in STOPWORDS or len(w) <= 2]
    return ' '.join(tokens)

# Apply cleaning
print("Cleaning text columns...")
for col in ['title', 'description', 'company_profile', 'requirements', 'benefits']:
    df[col] = df[col].apply(clean_text)
    print(f"  ✓ {col}")

print("\nSample cleaned description:")
print(df['description'].iloc[0][:300])

In [ ]:
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

# ── 1. TEXT FEATURES ──────────────────────────────────────────

# Combine the most informative text columns
df['combined_text'] = (
    df['title'].fillna('') + ' ' +
    df['description'].fillna('') + ' ' +
    df['requirements'].fillna('')
)

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),   # unigrams + bigrams
    sublinear_tf=True,    # dampens very high term frequencies
    min_df=3,             # ignore very rare terms
    max_df=0.95           # ignore near-universal terms
)

print("Fitting TF-IDF...")
tfidf_features = tfidf.fit_transform(df['combined_text'])
print(f"TF-IDF matrix shape: {tfidf_features.shape}")

# ── 2. METADATA FEATURES ─────────────────────────────────────

def build_metadata(df):
    """Engineer fraud-signal metadata features."""
    m = pd.DataFrame()

    # Binary flags — missing values are fraud signals
    m['has_company_logo']     = df['has_company_logo'].fillna(0).astype(int)
    m['telecommuting']        = df['telecommuting'].fillna(0).astype(int)
    m['missing_salary']       = df['salary_range'].isna().astype(int)
    m['missing_company']      = (df['company_profile'] == '').astype(int)
    m['missing_requirements'] = (df['requirements'] == '').astype(int)
    m['missing_benefits']     = (df['benefits'] == '').astype(int)

    # Text length features
    m['desc_len']         = df['description'].str.len().fillna(0)
    m['title_len']        = df['title'].str.len().fillna(0)
    m['req_len']          = df['requirements'].str.len().fillna(0)

    # Excitement/urgency signals  
    m['exclamation_count'] = df['description'].str.count('!').fillna(0)
    m['question_count']    = df['description'].str.count('\?').fillna(0)

    # Uppercase ratio (SHOUTING = scam signal)
    def upper_ratio(text):
        if not text or len(text) == 0:
            return 0
        upper = sum(1 for c in str(text) if c.isupper())
        return upper / max(len(str(text)), 1)
    m['upper_ratio'] = df['description'].apply(upper_ratio)

    # Money/earn keywords in title
    money_words = ['earn', 'income', 'salary', '\$', 'paid', 'weekly', 'daily', 'cash']
    pattern = '|'.join(money_words)
    m['money_in_title'] = df['title'].str.lower().str.contains(pattern, na=False).astype(int)

    # Employment type encoding
    m['is_parttime'] = (df['employment_type'] == 'Part-time').astype(int)
    m['is_contract'] = (df['employment_type'] == 'Contract').astype(int)

    return m

meta = build_metadata(df)
print(f"Metadata features: {meta.shape[1]} columns")
print(list(meta.columns))

In [ ]:
# ── 3. COMBINE ALL FEATURES ───────────────────────────────────
meta_sparse = csr_matrix(meta.values.astype(float))
X = hstack([tfidf_features, meta_sparse])
y = df['fraudulent'].values

print(f"Final feature matrix: {X.shape}")
print(f"Label distribution: {np.bincount(y)}")

# ── 4. TRAIN/TEST SPLIT (stratified to preserve class ratio) ──
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train fraud rate: {y_train.mean()*100:.1f}%")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("Training Logistic Regression...")
lr = LogisticRegression(
    C=1.0,
    class_weight='balanced',  # handles class imbalance
    max_iter=1000,
    solver='saga',
    n_jobs=-1
)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

print("\n── Logistic Regression ──")
print(classification_report(y_test, y_pred_lr, target_names=['Real', 'Fake']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_lr):.4f}")

### Model 2: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

print("Training Random Forest (this takes ~3–5 min)...")
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("\n── Random Forest ──")
print(classification_report(y_test, y_pred_rf, target_names=['Real', 'Fake']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_rf):.4f}")

### Model 3: XGBoost

In [ ]:
from xgboost import XGBClassifier

# Calculate scale_pos_weight for imbalanced classes
neg, pos = np.bincount(y_train)
scale = neg / pos
print(f"scale_pos_weight = {scale:.1f}")

print("Training XGBoost...")
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale,  # handles class imbalance
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1,
    tree_method='hist'  # faster on Colab
)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=50)
y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

print("\n── XGBoost ──")
print(classification_report(y_test, y_pred_xgb, target_names=['Real', 'Fake']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_xgb):.4f}")

### Confusion Matrices + ROC Comparison

In [ ]:
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

models = [
    ('Logistic Reg', y_pred_lr, y_prob_lr, '#3498db'),
    ('Random Forest', y_pred_rf, y_prob_rf, '#2ecc71'),
    ('XGBoost', y_pred_xgb, y_prob_xgb, '#e67e22'),
]

# Confusion matrices
for i, (name, y_pred, _, color) in enumerate(models):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
    axes[i].set_title(f'{name}\nF1-Fraud: {2*cm[1,1]/(2*cm[1,1]+cm[0,1]+cm[1,0]):.3f}', fontweight='bold')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

# ROC curves
for name, _, y_prob, color in models:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[3].plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.3f})')

axes[3].plot([0,1],[0,1],'k--', alpha=0.4)
axes[3].set_xlabel('False Positive Rate')
axes[3].set_ylabel('True Positive Rate')
axes[3].set_title('ROC Curves', fontweight='bold')
axes[3].legend(loc='lower right', fontsize=9)

plt.suptitle('Model Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: model_comparison.png")

### SHAP Explainability (on XGBoost — best performer)

In [ ]:
import shap

print("Computing SHAP values (this takes ~2 min)...")

# Use a sample for speed (full set is slow)
sample_idx = np.random.choice(X_test.shape[0], size=500, replace=False)
X_test_sample = X_test[sample_idx]

# Get feature names
tfidf_names = tfidf.get_feature_names_out().tolist()
meta_names = list(meta.columns)
all_feature_names = tfidf_names + meta_names

# Dense sample for SHAP
X_test_dense = X_test_sample.toarray()

explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test_dense)

# SHAP Summary Plot
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values, X_test_dense,
    feature_names=all_feature_names,
    max_display=20,
    show=False
)
plt.title('SHAP Feature Importance — Top 20 Features', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: shap_summary.png")

In [ ]:
# SHAP Bar plot — mean absolute impact
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values, X_test_dense,
    feature_names=all_feature_names,
    max_display=15,
    plot_type='bar',
    show=False
)
plt.title('Mean SHAP Values — Feature Importance', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

### Optional: BERT Fine-tuning (requires GPU runtime)



In [ ]:
# Check if GPU is available
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU — switch runtime to T4 for BERT fine-tuning")

In [ ]:
# ── BERT FINE-TUNING (only run with GPU) ──────────────────────
# Skip this cell if torch.cuda.is_available() returned False

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from datasets import Dataset
from sklearn.metrics import f1_score

MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Use only description column (fits token limit)
train_df = df.iloc[X_train.indices if hasattr(X_train, 'indices') else range(len(df))]

# Create train/test split on original df indices
train_idx, test_idx, _, _ = train_test_split(
    range(len(df)), df['fraudulent'], test_size=0.2, random_state=42, stratify=df['fraudulent']
)
train_texts = df['description'].iloc[train_idx].tolist()
test_texts  = df['description'].iloc[test_idx].tolist()
train_labels = df['fraudulent'].iloc[train_idx].tolist()
test_labels  = df['fraudulent'].iloc[test_idx].tolist()

def tokenize(examples):
    return tokenizer(
        examples['text'],
        truncation=True, padding='max_length', max_length=256
    )

train_dataset = Dataset.from_dict({'text': train_texts, 'label': train_labels})
test_dataset  = Dataset.from_dict({'text': test_texts,  'label': test_labels})
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'f1': f1_score(labels, preds, pos_label=1)}

training_args = TrainingArguments(
    output_dir='./bert_results',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=200,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    fp16=torch.cuda.is_available(),
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("Training DistilBERT...")
trainer.train()

# Final evaluation
results = trainer.evaluate()
print(f"\n✅ BERT F1 on fraud class: {results['eval_f1']:.4f}")

# Save model
model.save_pretrained('./bert_fake_job_model')
tokenizer.save_pretrained('./bert_fake_job_model')
print("Model saved to ./bert_fake_job_model")

### Save the Best Model for Deployment

In [ ]:
import pickle

# Save XGBoost model + TF-IDF + metadata column list
artifacts = {
    'model': xgb,
    'tfidf': tfidf,
    'meta_columns': list(meta.columns),
}

with open('fake_job_artifacts.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

print("✅ Saved: fake_job_artifacts.pkl")
print("Download this file and place it in your Streamlit app folder.")

# Also download it
from google.colab import files
files.download('fake_job_artifacts.pkl')
files.download('model_comparison.png')
files.download('shap_summary.png')

In [ ]:
app_code = '''
import streamlit as st
import pickle
import numpy as np
import pandas as pd
import re
import shap
from scipy.sparse import hstack, csr_matrix

st.set_page_config(
    page_title="Fake Job Detector",
    page_icon="🔍",
    layout="centered"
)

@st.cache_resource
def load_artifacts():
    with open("fake_job_artifacts.pkl", "rb") as f:
        return pickle.load(f)

artifacts = load_artifacts()
model = artifacts["model"]
tfidf = artifacts["tfidf"]

def clean_text(text):
    if not text:
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http\\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\\s!?]", " ", text)
    return re.sub(r"\\s+", " ", text).lower().strip()

def extract_meta(title, description):
    return np.array([[
        0,  # has_company_logo — unknown from text
        0,  # telecommuting
        1,  # missing_salary — we don\'t have it
        0,  # missing_company
        0,  # missing_requirements
        0,  # missing_benefits
        len(description),
        len(title),
        0,
        description.count("!"),
        description.count("?"),
        sum(1 for c in description if c.isupper()) / max(len(description), 1),
        int(bool(re.search(r"earn|income|\\$|paid|weekly|daily|cash", title.lower()))),
        0,  # is_parttime
        0,  # is_contract
    ]])

def predict(title, description):
    combined = clean_text(title) + " " + clean_text(description)
    tfidf_vec = tfidf.transform([combined])
    meta_vec = csr_matrix(extract_meta(title, description))
    X = hstack([tfidf_vec, meta_vec])
    prob = model.predict_proba(X)[0][1]
    label = "🚨 LIKELY FAKE" if prob > 0.5 else "✅ LIKELY REAL"
    return label, prob

def highlight_suspicious(text):
    suspicious = [
        r"work from home", r"earn (\\$[\\d,]+|money)",
        r"no experience", r"immediate(ly)?", r"apply now",
        r"limited positions", r"guaranteed", r"wire transfer",
        r"weekly pay", r"\\$(\\d{3,})"
    ]
    highlighted = text
    for pattern in suspicious:
        highlighted = re.sub(
            pattern,
            lambda m: f"**:red[{m.group()}]**",
            highlighted, flags=re.IGNORECASE
        )
    return highlighted

# ── UI ─────────────────────────────────────────────────────────
st.title("🔍 Fake Job Posting Detector")
st.caption("Trained on EMSCAD dataset · 17,880 job postings · XGBoost + TF-IDF")
st.divider()

title = st.text_input("Job Title", placeholder="e.g. Data Entry Specialist – Work From Home")
description = st.text_area(
    "Job Description",
    placeholder="Paste the full job description here...",
    height=250
)

if st.button("Analyze Job Posting", type="primary", use_container_width=True):
    if not description.strip():
        st.warning("Please paste a job description.")
    else:
        with st.spinner("Analyzing..."):
            label, prob = predict(title, description)

        col1, col2 = st.columns([1, 1])
        with col1:
            st.metric("Verdict", label)
        with col2:
            st.metric("Fraud Probability", f"{prob*100:.1f}%")

        st.progress(float(prob), text=f"Confidence: {prob*100:.1f}% fraudulent")

        if prob > 0.5:
            st.error("⚠️ This posting shows signs of fraud. Do not share personal or financial information.")
        else:
            st.success("This posting appears legitimate, but always research the company independently.")

        # Highlight suspicious phrases
        st.subheader("Suspicious Phrase Detection")
        highlighted = highlight_suspicious(description)
        st.markdown(highlighted)

st.divider()
st.caption("Built with XGBoost · TF-IDF · SHAP · Streamlit")
'''

with open('app.py', 'w') as f:
    f.write(app_code.strip())

# Write requirements.txt
requirements = """streamlit
scikit-learn
xgboost
shap
pandas
numpy
scipy
"""
with open('requirements.txt', 'w') as f:
    f.write(requirements)

print("✅ app.py and requirements.txt written!")

from google.colab import files
files.download('app.py')
files.download('requirements.txt')
print("\nDownloaded both files. Deploy steps:")
print("1. Create a GitHub repo")
print("2. Push: app.py, requirements.txt, fake_job_artifacts.pkl")
print("3. Go to https://share.streamlit.io → New app → paste your repo URL")

---
##  Bonus: Word Cloud of Fraud vs Real Postings

In [ ]:
from wordcloud import WordCloud

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, label, color, title_text in [
    (axes[0], 0, 'Blues', 'Real Job Postings'),
    (axes[1], 1, 'Reds',  'Fake Job Postings'),
]:
    text = ' '.join(df[df['fraudulent'] == label]['description'].tolist())
    wc = WordCloud(
        width=800, height=400,
        background_color='white',
        colormap=color,
        max_words=100,
        collocations=False
    ).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title_text, fontsize=14, fontweight='bold')

plt.suptitle('Most Common Words: Real vs Fake Job Postings', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('wordcloud_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: wordcloud_comparison.png")

---


**Expected metrics (XGBoost):**
- Accuracy: ~98%
- F1 (fraud class): ~0.82–0.86
- ROC-AUC: ~0.97